# Apache Airflow Deep Dive for Data Engineers
## DAG Authoring, Orchestration Patterns & Production Operations

**What you'll learn:**
- Airflow architecture: Scheduler, Webserver, Workers, Metadata DB
- DAGs: structure, dependencies, scheduling, catchup, backfill
- Operators: BashOperator, PythonOperator, SparkSubmitOperator, custom
- Sensors: waiting for files, APIs, partitions
- XComs: passing data between tasks
- Connections & Variables: secrets management
- TaskFlow API (modern Python-native DAGs)
- Branching, triggering, and dynamic DAGs
- Testing and debugging DAGs locally
- Deployment: Docker, Kubernetes, MWAA (AWS), Cloud Composer (GCP)
- Production patterns: idempotency, SLA, alerting, retry
- Airflow vs Databricks Workflows vs Dagster vs Prefect
- Interview questions

**Prerequisites:** 16_orchestration_workflows.ipynb, 49_linux_for_de.ipynb
**Estimated Time:** 7-9 hours

---

> Airflow is the most widely used orchestration tool in data engineering.
> Even if your company uses Databricks Workflows or Dagster, understanding
> Airflow is essential -- it's in ~70% of DE job descriptions.

---

---
# Section 1: Airflow Architecture

## Core Components

```
┌─────────────────────────────────────────────────────────────────┐
│                        AIRFLOW CLUSTER                           │
│                                                                  │
│  ┌──────────────┐    ┌──────────────┐    ┌──────────────┐      │
│  │  SCHEDULER   │    │  WEBSERVER   │    │   WORKERS    │      │
│  │              │    │              │    │              │      │
│  │ - Parses DAGs│    │ - UI/API     │    │ - Execute    │      │
│  │ - Schedules  │    │ - Monitor    │    │   tasks      │      │
│  │   task runs  │    │ - Trigger    │    │ - Report     │      │
│  │ - Monitors   │    │   DAGs       │    │   status     │      │
│  │   state      │    │              │    │              │      │
│  └──────┬───────┘    └──────────────┘    └──────┬───────┘      │
│         │                                        │               │
│         └──────────┐  ┌──────────────────────────┘               │
│                    │  │                                           │
│               ┌────▼──▼────┐         ┌──────────────┐           │
│               │ METADATA   │         │  EXECUTOR    │           │
│               │ DATABASE   │         │              │           │
│               │ (Postgres) │         │ Local/Celery │           │
│               │            │         │ /Kubernetes  │           │
│               │ - DAG runs │         │              │           │
│               │ - Task state│         └──────────────┘           │
│               │ - XComs    │                                     │
│               │ - Variables│                                     │
│               └────────────┘                                     │
│                                                                  │
│  DAG FILES (Python): /dags/daily_etl.py, /dags/hourly_ingest.py │
└─────────────────────────────────────────────────────────────────┘
```

## Executor Types

| Executor | How Tasks Run | Use When |
|----------|--------------|----------|
| **SequentialExecutor** | One task at a time (same process) | Local dev only |
| **LocalExecutor** | Parallel on one machine (subprocess) | Small teams, dev/staging |
| **CeleryExecutor** | Distributed workers (Redis/RabbitMQ) | Production, horizontal scale |
| **KubernetesExecutor** | Each task = K8s pod | Cloud-native, isolation, auto-scale |

## DAG Lifecycle

```
1. Scheduler PARSES DAG files (every min by default)
2. Scheduler creates DAG_RUN for each scheduled interval
3. Scheduler creates TASK_INSTANCES for each task
4. Tasks are queued → Executor picks them up
5. Worker executes task → reports success/failure
6. Downstream tasks become eligible when dependencies met
7. DAG_RUN marked success when all tasks complete
```

In [ ]:
# Airflow DAG examples (conceptual -- requires Airflow runtime)
print("AIRFLOW DAG AUTHORING")
print("=" * 60)

print("""
# ================================================================
# EXAMPLE 1: Classic DAG (traditional style)
# ================================================================
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from airflow.providers.apache.spark.operators.spark_submit import SparkSubmitOperator
from airflow.sensors.filesystem import FileSensor
from datetime import datetime, timedelta

default_args = {
    'owner': 'data_engineering',
    'depends_on_past': False,
    'email': ['alerts@company.com'],
    'email_on_failure': True,
    'retries': 3,
    'retry_delay': timedelta(minutes=5),
    'retry_exponential_backoff': True,
    'execution_timeout': timedelta(hours=2),
}

with DAG(
    dag_id='daily_orders_pipeline',
    default_args=default_args,
    description='Daily orders ETL: ingest -> transform -> gold',
    schedule='0 2 * * *',             # Run at 2 AM daily
    start_date=datetime(2024, 1, 1),
    catchup=False,                     # Don't backfill old runs
    max_active_runs=1,                 # Only 1 run at a time
    tags=['production', 'orders'],
) as dag:

    # Task 1: Wait for source file
    wait_for_data = FileSensor(
        task_id='wait_for_source_file',
        filepath='/data/incoming/orders_{{ ds }}.csv',
        poke_interval=300,      # Check every 5 minutes
        timeout=3600,           # Give up after 1 hour
        mode='poke',            # Or 'reschedule' to free worker
    )

    # Task 2: Ingest raw data
    ingest = SparkSubmitOperator(
        task_id='ingest_to_bronze',
        application='/opt/pipelines/ingest_orders.py',
        conf={'spark.sql.shuffle.partitions': '200'},
        application_args=['--date', '{{ ds }}'],
    )

    # Task 3: Transform (Python function)
    def transform_orders(**context):
        execution_date = context['ds']
        # Run transformation logic
        print(f"Transforming orders for {execution_date}")
        return f"Transformed {execution_date}"

    transform = PythonOperator(
        task_id='transform_to_silver',
        python_callable=transform_orders,
    )

    # Task 4: Build gold layer
    build_gold = BashOperator(
        task_id='build_gold_tables',
        bash_command='spark-submit /opt/pipelines/build_gold.py --date {{ ds }}',
    )

    # Task 5: Quality check
    quality_check = PythonOperator(
        task_id='run_quality_checks',
        python_callable=lambda: print("Quality checks passed!"),
    )

    # Define dependencies (execution order)
    wait_for_data >> ingest >> transform >> build_gold >> quality_check
""")

print("KEY CONCEPTS:")
print("  {{ ds }}: Airflow template for execution date (YYYY-MM-DD)")
print("  >> operator: defines dependency (run left before right)")
print("  default_args: inherited by all tasks (retries, timeout)")
print("  schedule='0 2 * * *': cron expression (2 AM daily)")
print("  catchup=False: don't run for past dates on deploy")

In [ ]:
# Modern TaskFlow API (Airflow 2.0+)
print("\nTASKFLOW API (Modern Python-Native DAGs)")
print("=" * 60)

print("""
# ================================================================
# EXAMPLE 2: TaskFlow API (recommended for new DAGs)
# ================================================================
from airflow.decorators import dag, task
from datetime import datetime

@dag(
    dag_id='orders_etl_taskflow',
    schedule='0 2 * * *',
    start_date=datetime(2024, 1, 1),
    catchup=False,
    tags=['production'],
)
def orders_etl():

    @task()
    def extract(date: str) -> dict:
        """Extract orders for given date."""
        print(f"Extracting orders for {date}")
        return {"records": 15000, "source": "postgres", "date": date}

    @task()
    def transform(raw_data: dict) -> dict:
        """Clean and transform extracted data."""
        print(f"Transforming {raw_data['records']} records")
        return {"records_clean": 14500, "records_quarantined": 500}

    @task()
    def load(transform_result: dict):
        """Load transformed data to gold layer."""
        print(f"Loading {transform_result['records_clean']} clean records to Delta")

    @task()
    def notify(transform_result: dict):
        """Send completion notification."""
        print(f"Pipeline complete: {transform_result['records_clean']} loaded")

    # Define flow (XCom passing is AUTOMATIC!)
    raw = extract(date="{{ ds }}")
    cleaned = transform(raw)
    load(cleaned)
    notify(cleaned)

# Instantiate the DAG
orders_etl()
""")

print("TASKFLOW BENEFITS:")
print("  - Pure Python functions (no operator boilerplate)")
print("  - XCom passing is automatic (return values flow)")
print("  - Type hints work for IDE support")
print("  - Easier to test (just call the function!)")
print("  - Cleaner than PythonOperator + callable pattern")

---
# Section 3: Key Airflow Concepts

## Operators, Sensors, and Hooks

| Type | Purpose | Examples |
|------|---------|---------|
| **Operators** | Execute a task | BashOperator, PythonOperator, SparkSubmitOperator |
| **Sensors** | Wait for a condition | FileSensor, S3KeySensor, SqlSensor, HttpSensor |
| **Hooks** | Connect to external systems | PostgresHook, S3Hook, SparkHook |
| **Transfers** | Move data between systems | S3ToRedshiftOperator, GCSToS3Operator |

## Scheduling & Execution Date

```
schedule='0 2 * * *' means "run at 2 AM daily"

IMPORTANT: execution_date is the START of the interval, NOT when it runs!

  Interval:     [2024-01-15 00:00] ──────────── [2024-01-16 00:00]
  execution_date: 2024-01-15
  Actually runs at: 2024-01-16 02:00  (END of interval + schedule time)

  This confuses EVERYONE at first. Think of it as:
  "Process data FOR this date" not "run ON this date"
```

## XCom (Cross-Communication)

```python
# Push data from one task
def extract(**context):
    result = {"count": 1000, "file": "/data/output.parquet"}
    return result  # Automatically pushed to XCom

# Pull in another task
def transform(**context):
    ti = context['task_instance']
    extract_result = ti.xcom_pull(task_ids='extract')
    print(f"Processing {extract_result['count']} records")

# LIMITATION: XCom stores in metadata DB (not for large data!)
# Max: ~48KB (Postgres) or ~64KB (MySQL)
# For large data: write to S3/GCS and pass the PATH via XCom
```

## Connections & Variables

```python
# Connections: store credentials in Airflow's encrypted store
# Set via UI: Admin > Connections
# Or CLI: airflow connections add 'postgres_prod' --conn-uri 'postgresql://...'

from airflow.hooks.base import BaseHook
conn = BaseHook.get_connection('postgres_prod')
print(f"Host: {conn.host}, Port: {conn.port}")

# Variables: runtime configuration
from airflow.models import Variable
env = Variable.get("environment")  # "production"
config = Variable.get("pipeline_config", deserialize_json=True)
```

In [ ]:
# Production Airflow patterns
print("PRODUCTION AIRFLOW PATTERNS")
print("=" * 60)

print("""
1. IDEMPOTENT TASKS (the #1 rule):
   - Running a task TWICE produces the SAME result
   - Use MERGE/upsert instead of INSERT
   - Use overwrite mode for file writes
   - Include date partition in output path

   # BAD (not idempotent):
   INSERT INTO target SELECT * FROM source WHERE date = '{{ ds }}'
   # Running twice = DUPLICATE DATA!

   # GOOD (idempotent):
   MERGE INTO target USING source ON target.id = source.id ...
   # OR: Write to /output/date={{ ds }}/ with overwrite

2. BRANCHING (conditional paths):
   from airflow.operators.python import BranchPythonOperator

   def choose_path(**context):
       day = context['execution_date'].weekday()
       if day < 5:
           return 'weekday_processing'
       return 'weekend_processing'

   branch = BranchPythonOperator(
       task_id='check_day',
       python_callable=choose_path,
   )

3. DYNAMIC DAGS (generate tasks from config):
   tables = ['orders', 'customers', 'products']
   for table in tables:
       task = SparkSubmitOperator(
           task_id=f'process_{table}',
           application_args=['--table', table],
       )

4. SLA & ALERTING:
   with DAG(..., sla_miss_callback=alert_slack):
       task = PythonOperator(
           task_id='critical_task',
           sla=timedelta(hours=1),  # Must finish within 1 hour
       )

5. POOLS (limit concurrent resource usage):
   task = SparkSubmitOperator(
       task_id='heavy_spark_job',
       pool='spark_cluster',  # Max 5 concurrent Spark jobs
   )
""")

---
# Section 5: Airflow vs Alternatives

| Feature | Airflow | Databricks Workflows | Dagster | Prefect |
|---------|---------|---------------------|---------|---------|
| **Language** | Python DAGs | JSON/YAML + UI | Python | Python |
| **Scheduling** | Cron, dataset-triggered | Cron, file arrival | Cron, sensors | Cron, event-driven |
| **UI** | Good (Gantt, Graph, Logs) | Integrated with Databricks | Excellent (asset-focused) | Modern, cloud-native |
| **Scaling** | CeleryExecutor, K8s | Serverless (Databricks) | K8s native | Hybrid (cloud agents) |
| **Testing** | Possible but manual | Limited | First-class | Good |
| **Data awareness** | Limited (XCom) | Tight (Delta, UC) | Asset-centric | Artifacts |
| **Learning curve** | Moderate | Low (if on Databricks) | Moderate | Low |
| **Community** | Huge (most popular) | Growing | Growing fast | Growing |
| **Managed options** | MWAA (AWS), Composer (GCP), Astronomer | Native | Dagster Cloud | Prefect Cloud |

## When to Use Each

- **Airflow**: General orchestration, multi-team, multi-tool environment
- **Databricks Workflows**: All-in on Databricks, simple pipelines
- **Dagster**: Data-asset-centric teams, strong testing needs
- **Prefect**: Modern teams wanting simpler alternative to Airflow

In [ ]:
# Deployment patterns
print("AIRFLOW DEPLOYMENT OPTIONS")
print("=" * 60)

print("""
1. DOCKER COMPOSE (Development / Small Teams):
   docker-compose up -d
   # Runs scheduler, webserver, worker, postgres, redis
   # Good for: local dev, small production

2. KUBERNETES (Production at Scale):
   helm install airflow apache-airflow/airflow
   # KubernetesExecutor: each task = isolated pod
   # Auto-scaling, resource isolation, cloud-native
   # Good for: large teams, strict isolation needs

3. AWS MWAA (Managed Workflows for Apache Airflow):
   # Fully managed by AWS
   # S3 for DAG storage, CloudWatch for logs
   # Auto-scaling workers
   # Good for: AWS shops, no infrastructure management

4. GCP Cloud Composer:
   # Fully managed on GKE
   # Integrated with BigQuery, Dataflow, GCS
   # Good for: GCP shops

5. Astronomer (Managed Airflow):
   # Company behind Airflow maintenance
   # Best managed experience (Astro CLI for dev)
   # Good for: teams wanting premium support

PROJECT STRUCTURE (best practice):
  dags/
  ├── daily/
  │   ├── orders_etl.py
  │   └── customer_metrics.py
  ├── hourly/
  │   └── streaming_check.py
  ├── common/
  │   ├── operators/
  │   ├── hooks/
  │   └── utils/
  plugins/
  ├── custom_operators.py
  └── custom_sensors.py
  tests/
  ├── test_orders_etl.py
  └── test_customer_metrics.py
  requirements.txt
  Dockerfile
  docker-compose.yml
""")

In [ ]:
print("\nAIRFLOW INTERVIEW QUESTIONS")
print("=" * 60)

questions = [
    {
        "q": "What is the difference between execution_date and the actual run time?",
        "a": "execution_date is the logical date (start of the data interval being processed). The actual run time is execution_date + schedule_interval. Example: DAG scheduled daily, execution_date=Jan 15 means it RUNS on Jan 16 to process Jan 15's data. This is Airflow's most confusing concept."
    },
    {
        "q": "How do you make Airflow tasks idempotent?",
        "a": "Use MERGE/upsert instead of INSERT. Write to date-partitioned paths with overwrite. Include execution_date in output paths. Use 'IF NOT EXISTS' checks. Never rely on auto-incrementing IDs. Test by running the same task twice -- result should be identical."
    },
    {
        "q": "How do you handle secrets/credentials in Airflow?",
        "a": "Use Airflow Connections (encrypted in metadata DB) accessed via Hooks. Use Variables for non-sensitive config. For AWS: use IAM roles on the worker nodes. For enterprise: integrate with HashiCorp Vault or AWS Secrets Manager via secrets backends. NEVER hardcode credentials in DAG files."
    },
    {
        "q": "KubernetesExecutor vs CeleryExecutor: when to use each?",
        "a": "CeleryExecutor: persistent workers, lower task startup latency, good for many small fast tasks. KubernetesExecutor: each task gets isolated pod, better resource isolation, auto-scaling to zero, different resource requirements per task. K8s is better for mixed workloads (some tasks need 1GB, others need 64GB)."
    },
    {
        "q": "How do you handle DAG failures in production?",
        "a": "1) Set retries + exponential backoff in default_args. 2) email_on_failure for alerts. 3) SLA misses trigger callbacks. 4) Use on_failure_callback for custom alerting (Slack, PagerDuty). 5) Design for idempotency so manual re-runs are safe. 6) Use Airflow's 'Clear' feature to re-run failed tasks."
    },
    {
        "q": "What are the limitations of XCom?",
        "a": "XCom stores data in the metadata DB (Postgres/MySQL). Size limit: ~48KB. Not suitable for passing DataFrames or large datasets. Solution: write data to S3/GCS/Delta Lake and pass the PATH via XCom. TaskFlow API makes this pattern cleaner with automatic serialization."
    },
]

for i, qa in enumerate(questions, 1):
    print(f"\nQ{i}: {qa['q']}")
    print(f"{'─'*60}")
    print(f"  {qa['a']}")

---
# Summary: Airflow Cheat Sheet

## DAG Writing Checklist

1. Set `catchup=False` unless you need backfill
2. Set `max_active_runs=1` for dependent pipelines
3. Use `default_args` for retries, timeouts, alerts
4. Make ALL tasks idempotent
5. Use Connections (not hardcoded credentials)
6. Use templates (`{{ ds }}`, `{{ execution_date }}`)
7. Keep DAG files simple (import heavy logic from modules)
8. Test DAGs before deploying (`airflow dags test`)

## Common Airflow CLI Commands

```bash
airflow dags list                    # List all DAGs
airflow dags trigger daily_etl       # Manual trigger
airflow tasks test daily_etl ingest 2024-01-15  # Test single task
airflow dags backfill daily_etl -s 2024-01-01 -e 2024-01-31
airflow connections list             # List connections
airflow variables get environment    # Get variable
```

## Airflow Template Variables

| Variable | Value | Example |
|----------|-------|---------|
| `{{ ds }}` | Execution date (YYYY-MM-DD) | 2024-01-15 |
| `{{ ds_nodash }}` | Date without dashes | 20240115 |
| `{{ execution_date }}` | Full datetime object | 2024-01-15T00:00:00 |
| `{{ prev_ds }}` | Previous execution date | 2024-01-14 |
| `{{ next_ds }}` | Next execution date | 2024-01-16 |
| `{{ params.key }}` | Custom parameters | user-defined |

---
*Apache Airflow Deep Dive for Data Engineers*